In [1]:
import pandas as pd
import json
import os
import folium
from folium import plugins
from branca.element import Template, MacroElement

In [ ]:
# ── 1. CARICAMENTO DATI ──────────────────────────────────────────────
BASE_DIR: "../../"
DATA_DIR = os.path.join(BASE_DIR,"indicatore_gem_score/output_gem_score")
# Immaginiamo tu abbia creato il nuovo CSV mensile
CSV_MENSILE_PATH = os.path.join(DATA_DIR, "indicatore_gem_score_mensile.csv")
GEOJSON_PATH = os.path.join(DATA_DIR, "comuni_sardegna.geojson")

# Per test, creiamo un dataset dummy se non esiste per dimostrare il funzionamento
if not os.path.exists(CSV_MENSILE_PATH):
    print("Creazione dataset mensile di test...")
    df_mensile = pd.DataFrame([
        # Esempio: Alghero è Q4 ad Agosto, ma Q1 a Gennaio
        {"codice_istat": "090003", "nome_comune": "Alghero", "anno": 2024, "mese": "Gennaio", "quadrante": "Q1", "gem_score_normalized": 85},
        {"codice_istat": "090003", "nome_comune": "Alghero", "anno": 2024, "mese": "Agosto", "quadrante": "Q4", "gem_score_normalized": 10},
        # Esempio: Bosa rimane Q1 sempre
        {"codice_istat": "091008", "nome_comune": "Bosa", "anno": 2024, "mese": "Gennaio", "quadrante": "Q1", "gem_score_normalized": 90},
        {"codice_istat": "091008", "nome_comune": "Bosa", "anno": 2024, "mese": "Agosto", "quadrante": "Q2", "gem_score_normalized": 75},
    ])
else:
    df_mensile = pd.read_csv(CSV_MENSILE_PATH)

df_mensile["codice_istat"] = df_mensile["codice_istat"].astype(str).str.zfill(6)

# ── 2. PREPARAZIONE GEOJSON CON DATI ANNIDATI ───────────────────────
with open(GEOJSON_PATH, "r") as f:
    geo_data = json.load(f)

# Strutturiamo i dati come un dizionario: dict[codice_istat][anno-mese] = {quadrante, score}
dati_per_comune = {}
for _, row in df_mensile.iterrows():
    cod = row["codice_istat"]
    if cod not in dati_per_comune:
        dati_per_comune[cod] = {}
    time_key = f"{row['anno']}-{row['mese']}"
    dati_per_comune[cod][time_key] = {
        "quadrante": row["quadrante"],
        "score": row["gem_score_normalized"]
    }

# Inseriamo i dati temporali direttamente nelle 'properties' di ciascun poligono GeoJSON
for feature in geo_data["features"]:
    # Assumiamo che il GeoJSON abbia una properties "PRO_COM" o "codice_istat"
    # Adatta il nome della chiave in base al tuo file!
    codice = str(feature["properties"].get("PRO_COM", feature["properties"].get("codice_istat", ""))).zfill(6)
    feature["properties"]["monthly_data"] = dati_per_comune.get(codice, {})
    # Impostiamo un quadrante di default per l'inizializzazione (es. 2024-Gennaio)
    feature["properties"]["quadrante_attuale"] = dati_per_comune.get(codice, {}).get("2024-Gennaio", {}).get("quadrante", "Q3")

# ── 3. CREAZIONE MAPPA FOLIUM ────────────────────────────────────────
m = folium.Map(location=[40.0, 9.5], zoom_start=7, tiles="CartoDB positron")

palette_hex = {"Q1": "#2ecc71", "Q2": "#e67e22", "Q3": "#3498db", "Q4": "#e74c3c"}

def style_fn(feature):
    q = feature["properties"].get("quadrante_attuale", "Q3")
    return {"fillColor": palette_hex.get(q, "#cccccc"), "color": "#333", "weight": 0.5, "fillOpacity": 0.7}

# Aggiungiamo il GeoJSON
geojson_layer = folium.GeoJson(
    geo_data,
    name="Gemme Nascoste Temporali",
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=["nome_comune"], aliases=["Comune:"])
).add_to(m)

# ── 4. INIEZIONE INTERFACCIA HTML E JAVASCRIPT (LA MASCHERA) ─────────
html_controls = """
{% macro html(this, kwargs) %}
<div style="position:fixed; top:20px; right:20px; z-index:9999; background:white; padding:15px; border-radius:8px; border:2px solid #ccc; box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
    <h4 style="margin-top:0;">Filtra Gemme del Mese</h4>
    <label for="sel-anno">Anno:</label>
    <select id="sel-anno" onchange="updateMap()">
        <option value="2024" selected>2024</option>
        <option value="2023">2023</option>
    </select>
    <br><br>
    <label for="sel-mese">Mese:</label>
    <select id="sel-mese" onchange="updateMap()">
        <option value="Gennaio" selected>Gennaio</option>
        <option value="Aprile">Aprile</option>
        <option value="Agosto">Agosto</option>
        <option value="Ottobre">Ottobre</option>
    </select>
</div>
{% endmacro %}
"""

js_script = """
{% macro script(this, kwargs) %}
function updateMap() {
    var anno = document.getElementById("sel-anno").value;
    var mese = document.getElementById("sel-mese").value;
    var timeKey = anno + "-" + mese;
    
    var mapInstance = {{this._parent.get_name()}};
    
    // Palette dei quadranti
    var palette = {"Q1": "#2ecc71", "Q2": "#e67e22", "Q3": "#3498db", "Q4": "#e74c3c"};

    // Itera su tutti i layer della mappa per trovare i poligoni GeoJSON
    mapInstance.eachLayer(function(layer) {
        if (layer.feature && layer.feature.properties && layer.feature.properties.monthly_data) {
            var mData = layer.feature.properties.monthly_data[timeKey];
            var newColor = "#cccccc"; // default se non c'è dato
            if (mData && mData.quadrante) {
                newColor = palette[mData.quadrante] || "#cccccc";
            }
            // Aggiorna il colore del layer
            layer.setStyle({fillColor: newColor});
        }
    });
}
{% endmacro %}
"""

# Applichiamo i template alla mappa
macro_html = MacroElement()
macro_html._template = Template(html_controls)
m.get_root().add_child(macro_html)

macro_js = MacroElement()
macro_js._template = Template(js_script)
m.get_root().add_child(macro_js)

# Aggiungiamo la legenda originale
legend_html = """
<div style="position:fixed;bottom:20px;left:20px;z-index:9999;font-size:13px;background:white;padding:10px;border-radius:8px;border:1px solid #aaa;">
<b>Quadranti</b><br>
<i style="background:#2ecc71;width:12px;height:12px;display:inline-block;margin-right:5px;"></i>🌟 Gemma Nascosta<br>
<i style="background:#e67e22;width:12px;height:12px;display:inline-block;margin-right:5px;"></i>🏖️ Destinazione Popolare<br>
<i style="background:#3498db;width:12px;height:12px;display:inline-block;margin-right:5px;"></i>🌄 Territorio Autentico<br>
<i style="background:#e74c3c;width:12px;height:12px;display:inline-block;margin-right:5px;"></i>⚠️ Zona Satura
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Salvataggio
out_path = os.path.join(DATA_DIR, "mappa_gemme_interattiva_mensile.html")
m.save(out_path)
print(f"✅ Mappa salvata → {out_path}")
display(m)

Creazione dataset mensile di test...


FileNotFoundError: [Errno 2] No such file or directory: 'data/sardegna-attrattivita/comuni_sardegna.geojson'